In [1]:
! git clone https://github.com/George-Malak/Aegis-AI-Violence-Detection-System.git

Cloning into 'Aegis-AI-Violence-Detection-System'...
remote: Enumerating objects: 362, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (273/273), done.
remote: Total 362 (delta 127), reused 249 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (362/362), 7.16 MiB | 18.89 MiB/s, done.
Resolving deltas: 100% (127/127), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import cv2
import os
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [17]:
def extract_dataset_info(base_dir):
    data_list = []

    splits = ['Train', 'Val', 'Test']

    for split in splits:
        split_path = os.path.join(base_dir, split)
        if not os.path.exists(split_path):
            continue

        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            if not os.path.isdir(class_path):
                continue

            for video_name in os.listdir(class_path):
                if video_name.endswith(('.mp4', '.avi', '.mkv', '.mov')):
                    video_path = os.path.join(class_path, video_name)

                    cap = cv2.VideoCapture(video_path)
                    fps = cap.get(cv2.CAP_PROP_FPS)
                    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
                    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
                    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
                    duration = frame_count / fps if fps > 0 else 0
                    cap.release()

                    source = 'Arabic' if 'arabic' in video_name.lower() else 'UCF'

                    data_list.append({
                        'Video_Name': video_name,
                        'Split': split,          # train, val, test
                        'Class': class_name,      # violence, non-violence
                        'Source': source,        # Arabic, UCF
                        'Duration_Sec': duration,
                        'Total_Frames': frame_count,
                        'FPS': fps,
                        'Resolution': f"{int(width)}x{int(height)}"
                    })

    return pd.DataFrame(data_list)

In [ ]:
ROOT_DIR = Path('/content/drive/MyDrive/DEPI Project').resolve()
df = extract_dataset_info(ROOT_DIR)

df.head()


In [ ]:
print("--- # of Videos of each stage ---")
print(df['Split'].value_counts())

In [9]:
print("\n--- Details of Classes ---")
print(df.groupby(['Split', 'Class', 'Source']).size().unstack(fill_value=0))

In [ ]:
plt.figure(figsize=(14, 6))
sns.set_theme(style="whitegrid")

g = sns.catplot(
    data=df, x="Split", hue="Source", col="Class",
    kind="count", palette="Set1", height=5, aspect=1.2
)
g.set_axis_labels("Dataset Split", "Number of Videos")
g.fig.suptitle("Video Count: Split vs Source grouped by Class", y=1.05)
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
sns.kdeplot(data=df, x="Duration_Sec", hue="Source", fill=True, common_norm=False, alpha=0.5, palette="Dark2")
plt.title("Video Duration Distribution: Arabic vs UCF")
plt.xlabel("Duration (Seconds)")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="Source", y="Total_Frames", palette="Pastel1")
plt.title("Total Frames Comparison: Arabic vs UCF")
plt.ylabel("Total Frames")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="FPS", hue="Source", palette="muted")
plt.title("FPS Distribution: Arabic vs UCF")
plt.xlabel("Frames Per Second (FPS)")
plt.ylabel("Video Count")
plt.show()